# CUSIP Statistics View

Creates `nyu-datasets.munibonds.cusip_statistics` — a per-CUSIP aggregate of trading activity used by other notebooks (`02_descriptive_analysis`, `03_baseline_pricing`) to filter the active universe.

Computed as a **view**, so it always reflects the latest data in `trades_typed` without manual refresh.

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project='nyu-datasets')
DATASET = 'nyu-datasets.munibonds'

## Schema of `cusip_statistics`

| Column | Type | Description |
|--------|------|-------------|
| `CUSIP` | STRING | Bond identifier |
| `SECURITY_DESCRIPTION` | STRING | Description (any value sample) |
| `MATURITY_DATE` | DATE | Bond maturity (min observed) |
| `DATED_DATE` | DATE | Issue dated date (min observed) |
| `COUPON` | FLOAT64 | Coupon rate (min observed) |
| `NUM_TRADES` | INT64 | Total trades |
| `D_TRADES` | INT64 | Dealer-to-dealer trades |
| `P_TRADES` | INT64 | Customer-purchase (dealer buys from customer) |
| `S_TRADES` | INT64 | Customer-sale (dealer sells to customer) |
| `NUM_DATES_TRADED` | INT64 | Distinct trade dates |
| `FIRST_TRADE` / `LAST_TRADE` | DATE | First / last observed trade date |
| `DAYS_AVAILABLE` | INT64 | `LAST_TRADE - FIRST_TRADE + 1` (calendar days, inclusive) |
| `FRACTION_DAYS_TRADED` | FLOAT64 | `NUM_DATES_TRADED / DAYS_AVAILABLE` (1.0 = traded every day) |
| `AVG_PRICE`, `AVG_YIELD` | FLOAT64 | Mean across all trades |
| `STDDEV_PRICE`, `STDDEV_YIELD` | FLOAT64 | Standard deviation |
| `NULL_PRICE_PCT`, `NULL_YIELD_PCT` | FLOAT64 | Fraction of trades with NULL price/yield |

In [ ]:
VIEW_QUERY = f"""
CREATE OR REPLACE VIEW `{DATASET}.cusip_statistics` AS
SELECT
  CUSIP,
  ANY_VALUE(SECURITY_DESCRIPTION) AS SECURITY_DESCRIPTION,
  MIN(MATURITY_DATE) AS MATURITY_DATE,
  MIN(DATED_DATE) AS DATED_DATE,
  MIN(COUPON) AS COUPON,
  COUNT(*) AS NUM_TRADES,
  COUNTIF(TRADE_TYPE_INDICATOR = 'D') AS D_TRADES,
  COUNTIF(TRADE_TYPE_INDICATOR = 'P') AS P_TRADES,
  COUNTIF(TRADE_TYPE_INDICATOR = 'S') AS S_TRADES,
  COUNT(DISTINCT TRADE_DATE) AS NUM_DATES_TRADED,
  MIN(TRADE_DATE) AS FIRST_TRADE,
  MAX(TRADE_DATE) AS LAST_TRADE,
  DATE_DIFF(MAX(TRADE_DATE), MIN(TRADE_DATE), DAY) + 1 AS DAYS_AVAILABLE,
  SAFE_DIVIDE(
    COUNT(DISTINCT TRADE_DATE),
    DATE_DIFF(MAX(TRADE_DATE), MIN(TRADE_DATE), DAY) + 1
  ) AS FRACTION_DAYS_TRADED,
  AVG(DOLLAR_PRICE) AS AVG_PRICE,
  STDDEV(DOLLAR_PRICE) AS STDDEV_PRICE,
  AVG(YIELD) AS AVG_YIELD,
  STDDEV(YIELD) AS STDDEV_YIELD,
  SAFE_DIVIDE(COUNTIF(DOLLAR_PRICE IS NULL), COUNT(*)) AS NULL_PRICE_PCT,
  SAFE_DIVIDE(COUNTIF(YIELD IS NULL), COUNT(*)) AS NULL_YIELD_PCT
FROM `{DATASET}.trades_typed`
WHERE CUSIP IS NOT NULL
  AND TRADE_DATE IS NOT NULL
GROUP BY CUSIP
"""

client.query(VIEW_QUERY).result()
print(f'✓ Created view {DATASET}.cusip_statistics')

## Sanity-check the view

In [ ]:
result = client.query(f"""
SELECT
  COUNT(*) AS num_cusips,
  COUNTIF(D_TRADES > 100 AND NUM_DATES_TRADED > 10 AND FRACTION_DAYS_TRADED > 0.1) AS active_universe_size,
  MIN(FIRST_TRADE) AS earliest_trade,
  MAX(LAST_TRADE)  AS latest_trade
FROM `{DATASET}.cusip_statistics`
""").to_dataframe()
result

In [ ]:
# Top-10 most-traded CUSIPs
client.query(f"""
SELECT CUSIP, SECURITY_DESCRIPTION, NUM_TRADES, NUM_DATES_TRADED,
       ROUND(FRACTION_DAYS_TRADED, 3) AS FRACTION_DAYS_TRADED
FROM `{DATASET}.cusip_statistics`
ORDER BY NUM_TRADES DESC
LIMIT 10
""").to_dataframe()